# Polydisperse Polymer Systems → LAMMPS

**"How do I generate a LAMMPS-ready polydisperse polymer system from GBigSMILES?"**

This guide extends the polydisperse polymer workflow to create a complete LAMMPS simulation system, including:

1. **System planning**: Sample a polydisperse chain ensemble from GBigSMILES
2. **Atomistic building**: Convert chain sequences into 3D atomistic structures
3. **Typifying**: Assign LAMMPS atom types using a force field
4. **Packing**: Pack chains into a periodic simulation box
5. **Export**: Write LAMMPS data and force field files

## What We Will Build

A LAMMPS-ready polydisperse polymer melt with:
- Schulz–Zimm molecular weight distribution
- Mixed ethylene oxide (EO) and styrene oxide (SO) monomers
- Realistic density (~1.0 g/cm³)
- Full topology (bonds, angles, dihedrals)
- OPLS-AA force field parameters

## Setup: Imports

In [1]:
from random import Random
from pathlib import Path

import numpy as np

import molpy as mp
from molpy.builder.polymer import PolymerBuilder
from molpy.builder.polymer.connectors import ReacterConnector
from molpy.builder.polymer.placer import Placer
from molpy.builder.polymer.sequence_generator import WeightedSequenceGenerator
from molpy.builder.polymer.system import (
    PolydisperseChainGenerator,
    SchulzZimmPolydisperse,
    SystemPlanner,
    create_polydisperse_from_ir,
)
from molpy.core.element import Element
from molpy.core.box import Box
from molpy.core.atomistic import Atomistic
from molpy.compute.rdkit import Generate3D
from molpy.adapter.rdkit import RDKitAdapter
from molpy.parser.smiles import parse_bigsmiles, bigsmilesir_to_monomer
from molpy.reacter import (
    select_port,
    select_c_neighbor,
    select_hydroxyl_group,
    select_hydroxyl_h_only,
    form_single_bond,
)
from molpy.reacter.template import TemplateReacter
from molpy.typifier.atomistic import OplsAtomisticTypifier
from molpy.io.data.lammps import LammpsDataWriter
from molpy.io.forcefield.lammps import LAMMPSForceFieldWriter
from molpy.pack import InsideBoxConstraint, Molpack

## Step 1: Define the Polydisperse System

We use a GBigSMILES string to declare:
- **Repeat units**: Two monomers (ethylene oxide and styrene oxide)
- **Distribution**: Schulz–Zimm with Mn=4800, Mw=6000
- **End groups**: Simple hydrogen termination
- **Target mass**: Total system mass (controls number of chains)

In [2]:
# GBigSMILES string for polydisperse copolymer
gbigsmiles = "{[<]OCCOCCOCCOCCO[>],[<]OCC(c1ccccc1)CO[>]}|schulz_zimm(4800, 6000)|[H].|5e7|"

# Parse GBigSMILES
ir = mp.parser.parse_gbigsmiles(gbigsmiles)
component_ir = ir.molecules[0]
mol_ir = component_ir.molecule
target_mass = float(component_ir.target_mass)

# Extract distribution
distribution_ir = None
for meta in mol_ir.stochastic_metadata:
    if meta.distribution:
        distribution_ir = meta.distribution
        break

print(f"✅ Parsed GBigSMILES: {distribution_ir.name} distribution")
print(f"   Target total mass: {target_mass:.2e} g/mol")

✅ Parsed GBigSMILES: schulz_zimm distribution
   Target total mass: 5.00e+07 g/mol


## Step 2: Build Monomer Library

Convert repeat units to atomistic monomers with:
- 3D coordinates (RDKit embedding)
- Reactive ports for polymerization
- Topology (bonds, angles, dihedrals)

In [3]:
def build_monomer(bigsmiles: str) -> Atomistic:
    """Build a monomer from BigSMILES string with 3D coordinates."""
    ir = parse_bigsmiles(bigsmiles)
    monomer = bigsmilesir_to_monomer(ir)
    
    # Generate 3D coordinates
    adapter = RDKitAdapter(internal=monomer)
    generate_3d = Generate3D(add_hydrogens=True, embed=True, optimize=True, update_internal=True)
    adapter = generate_3d(adapter)
    monomer = adapter.get_internal()
    
    # Generate topology
    monomer.get_topo(gen_angle=True, gen_dihe=True)
    
    # Assign atom IDs
    for idx, atom in enumerate(monomer.atoms):
        atom["id"] = idx + 1
    
    return monomer

# Extract repeat units and build monomers
repeat_units = mol_ir.structure.stochastic_objects[0].repeat_units
stoch_obj = mol_ir.structure.stochastic_objects[0]

from molpy.parser.smiles.converter import create_monomer_from_repeat_unit

monomer_structures = []
for unit in repeat_units:
    monomer = create_monomer_from_repeat_unit(unit, stoch_obj)
    if monomer is not None:
        monomer_structures.append(monomer)

# Build 3D structures for each monomer
library = {}
monomer_masses_list = []

for i, monomer_2d in enumerate(monomer_structures):
    # Convert to BigSMILES for rebuilding
    label = f"#{i}"
    
    # Build 3D structure
    adapter = RDKitAdapter(internal=monomer_2d)
    generate_3d = Generate3D(add_hydrogens=True, embed=True, optimize=True, update_internal=True)
    adapter = generate_3d(adapter)
    monomer_3d = adapter.get_internal()
    
    # Generate topology
    monomer_3d.get_topo(gen_angle=True, gen_dihe=True)
    
    # Assign atom IDs
    for idx, atom in enumerate(monomer_3d.atoms):
        atom["id"] = idx + 1
    
    library[label] = monomer_3d
    
    # Compute monomer mass
    total_mass = 0.0
    for atom in monomer_3d.atoms:
        element_symbol = atom.get("element") or atom.get("symbol")
        if element_symbol:
            try:
                elem = Element(element_symbol)
                total_mass += elem.mass
            except KeyError:
                pass
    monomer_masses_list.append(total_mass)

monomer_masses = {str(i): m for i, m in enumerate(monomer_masses_list)}
weights_dict = {str(i): 1.0 for i in range(len(monomer_structures))}

print(f"✅ Built {len(library)} monomers:")
for label, monomer in library.items():
    print(f"   {label}: {len(monomer.atoms)} atoms, MW={monomer_masses[label.strip('#')]:.1f} g/mol")

✅ Built 2 monomers:
   #0: 31 atoms, MW=194.2 g/mol
   #1: 29 atoms, MW=158.2 g/mol


## Step 3: Sample Polydisperse Chain Ensemble

Use `SystemPlanner` to generate a polydisperse chain ensemble:
- Sample chain molecular weights from Schulz–Zimm distribution
- Generate monomer sequences for each chain
- Track chain DP and mass

In [4]:
# Create distribution object
random_seed = 43
dist_obj = create_polydisperse_from_ir(distribution_ir, random_seed=random_seed)

# Create sequence generator
seq_gen = WeightedSequenceGenerator(monomer_weights=weights_dict)

# Create chain generator
chain_gen = PolydisperseChainGenerator(
    seq_generator=seq_gen,
    monomer_mass=monomer_masses,
    end_group_mass=0.0,
    distribution=dist_obj,
)

# Plan system
rng = Random(random_seed)
planner = SystemPlanner(
    chain_generator=chain_gen,
    target_total_mass=target_mass,
    max_rel_error=0.02,
)
system_plan = planner.plan_system(rng)
chains = system_plan.chains

# Compute statistics
molecular_weights = [chain.mass for chain in chains]
Mn = float(np.mean(molecular_weights))
Mw = float(np.sum(np.array(molecular_weights)**2) / np.sum(molecular_weights))
PDI = Mw / Mn

print(f"✅ Sampled {len(chains)} chains:")
print(f"   Mn = {Mn:.0f} g/mol")
print(f"   Mw = {Mw:.0f} g/mol")
print(f"   PDI = {PDI:.3f}")
print(f"   Total mass = {sum(molecular_weights):.2e} g/mol")

✅ Sampled 10590 chains:
   Mn = 4722 g/mol
   Mw = 5946 g/mol
   PDI = 1.259
   Total mass = 5.00e+07 g/mol


## Step 4: Build Atomistic Polymer Chains

Convert each chain sequence into a 3D atomistic structure using `PolymerBuilder`:
- Connect monomers via dehydration reaction
- Preserve topology through all connections
- Generate CGSmiles strings from chain sequences

In [6]:
# Set up polymer builder
template_reacter = TemplateReacter(
    name="dehydration",
    anchor_selector_left=select_c_neighbor,
    anchor_selector_right=select_port,
    leaving_selector_left=select_hydroxyl_group,
    leaving_selector_right=select_hydroxyl_h_only,
    bond_former=form_single_bond,
    radius=4,
)

connector = ReacterConnector(default=template_reacter)
placer = Placer()

builder = PolymerBuilder(
    library=library,
    connector=connector,
    placer=placer,
    typifier=None,  # Will typify later
)

# Build atomistic chains
atomistic_chains = []
print(f"\nBuilding {len(chains)} atomistic chains...")

for i, chain in enumerate(chains[:10]):  # Build first 10 chains for demo
    # Convert sequence to CGSmiles
    sequence_labels = [f"[#{idx}]" for idx in chain.sequence]
    cgsmiles = "{" + "".join(sequence_labels) + "}"
    
    # Build polymer
    result = builder.build(cgsmiles)
    polymer = result.polymer
    
    # Regenerate topology after building
    polymer.get_topo(gen_angle=True, gen_dihe=True)
    
    atomistic_chains.append(polymer)
    
    if (i + 1) % 5 == 0:
        print(f"   Built {i+1}/{len(chains[:10])} chains...")

print(f"✅ Built {len(atomistic_chains)} atomistic chains")
print(f"   Example chain: {len(atomistic_chains[0].atoms)} atoms, {len(atomistic_chains[0].bonds)} bonds")

TypeError: ReacterConnector.__init__() missing 1 required positional argument: 'port_map'

## Step 5: Typify with Force Field

Assign LAMMPS atom types using OPLS-AA force field:
- Load force field parameters
- Typify all atoms, bonds, angles, dihedrals
- Collect all unique types for export

In [ ]:
# Load force field
forcefield_path = "oplsaa.xml"
ff = mp.io.read_xml_forcefield(forcefield_path)
typifier = OplsAtomisticTypifier(ff, strict_typing=False)

print("Typifying chains...")
for i, polymer in enumerate(atomistic_chains):
    typifier.typify(polymer)
    if (i + 1) % 5 == 0:
        print(f"   Typified {i+1}/{len(atomistic_chains)} chains...")

print(f"✅ Typified {len(atomistic_chains)} chains")

## Step 6: Pack into Simulation Box

Use Packmol to pack chains into a periodic box:
- Calculate box size from target density
- Pack chains with overlap constraints
- Assign molecule IDs for LAMMPS

In [ ]:
def calculate_box_size(chains: list[Atomistic], target_density: float = 1.0) -> float:
    """Calculate box length (Å) from target density (g/cm³)."""
    total_mw = 0.0
    for chain in chains:
        for atom in chain.atoms:
            symbol = atom.get("symbol", "C")
            elem = Element(symbol.upper())
            total_mw += elem.mass
    
    NA = 6.022e23
    total_mass_g = total_mw / NA
    volume_cm3 = total_mass_g / target_density
    volume_angstrom3 = volume_cm3 * 1e24
    box_length = volume_angstrom3 ** (1.0 / 3.0)
    return box_length

# Calculate box size
target_density = 1.0  # g/cm³
box_length = calculate_box_size(atomistic_chains, target_density)

print(f"\n✅ Calculated box size: {box_length:.2f} Å")
print(f"   Target density: {target_density} g/cm³")
print(f"   Number of chains: {len(atomistic_chains)}")

# Prepare frames for packing
chain_frames = []
for polymer in atomistic_chains:
    frame = polymer.to_frame()
    # Normalize charge field
    if "charge" in frame["atoms"]:
        frame["atoms"]["q"] = frame["atoms"]["charge"]
    chain_frames.append(frame)

# Pack with Molpack
output_dir = Path("user-guide-output") / "06_polymer_polydisperse_lammps"
output_dir.mkdir(parents=True, exist_ok=True)

pack_workdir = output_dir / "packmol_work"
packer = Molpack(workdir=pack_workdir)

origin = np.array([0.0, 0.0, 0.0])
length = np.array([box_length, box_length, box_length])
box_constraint = InsideBoxConstraint(length=length, origin=origin)

print("\nPacking chains...")
for frame in chain_frames:
    packer.add_target(frame, number=1, constraint=box_constraint)

packed_frame = packer.optimize(max_steps=10000, seed=42)
print(f"✅ Packed system: {packed_frame['atoms'].nrows} atoms")

# Add box metadata
box = Box.cubic(length=box_length)
packed_frame.metadata["box"] = box

# Ensure charge fields exist
if "charge" in packed_frame["atoms"] and "q" not in packed_frame["atoms"]:
    packed_frame["atoms"]["q"] = packed_frame["atoms"]["charge"]
elif "q" in packed_frame["atoms"] and "charge" not in packed_frame["atoms"]:
    packed_frame["atoms"]["charge"] = packed_frame["atoms"]["q"]

## Step 7: Collect Types and Export to LAMMPS

Collect all unique types and write LAMMPS files:
- Collect atom, bond, angle, dihedral types
- Write force field parameters (`.ff` file)
- Write data file (`.data` file)

In [ ]:
def collect_types_from_frames(*frames) -> dict[str, set[str]]:
    """Collect all type names from multiple frames."""
    atom_types = set()
    bond_types = set()
    angle_types = set()
    dihedral_types = set()
    
    for frame in frames:
        if "atoms" in frame and "type" in frame["atoms"]:
            for atom_type in frame["atoms"]["type"]:
                if atom_type:
                    type_str = str(atom_type)
                    if not type_str.isdigit():
                        atom_types.add(type_str)
        
        if "bonds" in frame and "type" in frame["bonds"]:
            for bond_type in frame["bonds"]["type"]:
                if bond_type:
                    type_str = str(bond_type)
                    if not type_str.isdigit():
                        bond_types.add(type_str)
        
        if "angles" in frame and "type" in frame["angles"]:
            for angle_type in frame["angles"]["type"]:
                if angle_type:
                    type_str = str(angle_type)
                    if not type_str.isdigit():
                        angle_types.add(type_str)
        
        if "dihedrals" in frame and "type" in frame["dihedrals"]:
            for dihedral_type in frame["dihedrals"]["type"]:
                if dihedral_type:
                    type_str = str(dihedral_type)
                    if not type_str.isdigit():
                        dihedral_types.add(type_str)
    
    return {
        "atom_types": atom_types,
        "bond_types": bond_types,
        "angle_types": angle_types,
        "dihedral_types": dihedral_types,
    }

# Collect types
types_from_config = collect_types_from_frames(packed_frame)

packed_frame.metadata["type_labels"] = {
    "atom_types": sorted(list(types_from_config["atom_types"])),
    "bond_types": sorted(list(types_from_config["bond_types"])),
    "angle_types": sorted(list(types_from_config["angle_types"])),
    "dihedral_types": sorted(list(types_from_config["dihedral_types"])),
}

print(f"\n✅ Collected types:")
print(f"   Atom types: {len(types_from_config['atom_types'])}")
print(f"   Bond types: {len(types_from_config['bond_types'])}")
print(f"   Angle types: {len(types_from_config['angle_types'])}")
print(f"   Dihedral types: {len(types_from_config['dihedral_types'])}")

# Write force field file
ff_path = output_dir / "polydisperse_system.ff"
ff_writer = LAMMPSForceFieldWriter(ff_path)
ff_writer.write(
    ff,
    atom_types=types_from_config["atom_types"],
    bond_types=types_from_config["bond_types"],
    angle_types=types_from_config["angle_types"],
    dihedral_types=types_from_config["dihedral_types"],
)

# Write data file
data_path = output_dir / "polydisperse_system.data"
data_writer = LammpsDataWriter(data_path, atom_style="full")
data_writer.write(packed_frame)

print(f"\n✅ Written LAMMPS files:")
print(f"   Force field: {ff_path}")
print(f"   Data file: {data_path}")

## Step 8: Generate LAMMPS Input Script

Create a runnable LAMMPS input script for equilibration and production MD.

In [ ]:
script_content = f"""# LAMMPS input script for polydisperse polymer system
# Force field: OPLS-AA
# Distribution: Schulz-Zimm (Mn={Mn:.0f}, Mw={Mw:.0f}, PDI={PDI:.3f})
# Number of chains: {len(atomistic_chains)}
#
units           real
atom_style      full
boundary        p p p
dimension       3

read_data       polydisperse_system.data
include         polydisperse_system.ff
kspace_style    pppm 1.0e-5
special_bonds   lj/coul 0.0 0.0 0.5

neighbor        2.0 bin
neigh_modify    delay 0 every 1 check yes

print "=========================================="
print "Step 1: Energy Minimization"
print "=========================================="
minimize        1.0e-4 1.0e-4 1000 10000

print "=========================================="
print "Step 2: NVT Equilibration"
print "=========================================="
velocity        all create 300.0 12345
timestep        1.0
fix             nvt all nvt temp 300.0 300.0 100.0
thermo          100
thermo_style    custom step temp pe ke etotal press vol density
dump            1 all custom 1000 equil_nvt.lammpstrj id mol type x y z
run             10000
undump          1
unfix           nvt

print "=========================================="
print "Step 3: NPT Equilibration"
print "=========================================="
fix             npt all npt temp 300.0 300.0 100.0 iso 1.0 1.0 1000.0
dump            2 all custom 1000 equil_npt.lammpstrj id mol type x y z
run             10000
undump          2
unfix           npt

print "=========================================="
print "Step 4: Production MD"
print "=========================================="
fix             npt_prod all npt temp 300.0 300.0 100.0 iso 1.0 1.0 1000.0
dump            3 all custom 1000 production.lammpstrj id mol type xu yu zu
run             50000
undump          3
unfix           npt_prod

write_data      final_system.data
write_restart   final_system.restart

print "=========================================="
print "Simulation completed!"
print "=========================================="
"""

script_path = output_dir / "run_polydisperse.in"
with script_path.open("w") as f:
    f.write(script_content)

print(f"✅ Generated LAMMPS input script: {script_path}")

## Summary

You now have a complete workflow for generating LAMMPS-ready polydisperse polymer systems:

1. ✅ **System planning**: Sampled polydisperse chain ensemble from GBigSMILES
2. ✅ **Monomer library**: Built 3D atomistic monomers with reactive ports
3. ✅ **Chain building**: Converted sequences to atomistic polymers via `PolymerBuilder`
4. ✅ **Typifying**: Assigned OPLS-AA force field types
5. ✅ **Packing**: Packed chains into periodic box with Packmol
6. ✅ **Export**: Generated LAMMPS data, force field, and input files

### Generated Files

All files are in `user-guide-output/06_polymer_polydisperse_lammps/`:

- `polydisperse_system.data` - LAMMPS data file (atom coordinates, topology)
- `polydisperse_system.ff` - Force field parameters
- `run_polydisperse.in` - LAMMPS input script

### Running the Simulation

```bash
cd user-guide-output/06_polymer_polydisperse_lammps
lmp -in run_polydisperse.in
```

### Next Steps

- **Adjust distribution**: Change Mn/Mw in GBigSMILES to explore different polydispersities
- **Add more chains**: Increase target mass to generate larger systems
- **Different monomers**: Modify repeat units in GBigSMILES
- **Analysis**: Use LAMMPS output to analyze Rg, end-to-end distance, etc.